## Stage 2c Revised: Continue Training the Baseline Adapter

This revision is intentionally conservative:
- keep the Stage 1 direct-answer format unchanged
- continue training the existing baseline LoRA adapter directly
- mix in cleaned CoT samples without `<think>` tags
- avoid packing because flash attention is unavailable in this environment


In [ ]:
!pip install -q --no-index --find-links /kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages trl peft datasets


In [ ]:
import os, sys, stat, shutil, gc, zipfile, json, re, random
import pandas as pd
import torch
import torch.nn.functional as F
import kagglehub
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from trl import SFTTrainer, SFTConfig
import types

SEED = 42
random.seed(SEED)

# --- Kaggle / Triton Environment Fixes (same spirit as Stage 1) ---
def purermsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                   group_size=None, norm_before_gate=True, upcast=True):
    del group_size, norm_before_gate
    dtype = x.dtype
    if upcast:
        x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None:
        out = out + bias.float()
    if z is not None:
        out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = purermsnorm_fn

src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell'
dst = '/tmp/ptxas-blackwell'
if os.path.exists(src):
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    import triton.backends.nvidia as nv_backend
    import importlib.util
    spec = importlib.util.find_spec('triton.backends.nvidia')
    real_nv_file = spec.origin if spec and spec.origin else None
    if real_nv_file:
        real_nv_dir = os.path.dirname(real_nv_file)
        src_bin = os.path.join(real_nv_dir, 'bin')
        if os.path.isdir(src_bin):
            dst_bin = '/tmp/triton_nvidia_bin'
            shutil.rmtree(dst_bin, ignore_errors=True)
            shutil.copytree(src_bin, dst_bin)
            for f in os.listdir(dst_bin):
                fp = os.path.join(dst_bin, f)
                if os.path.isfile(fp):
                    os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
            nv_backend.__file__ = os.path.join(dst_bin, '__init__.py')
            os.environ['TRITON_PTXAS_PATH'] = dst
            print('[INFO] Triton PTXAS fix applied.')

# --- Hyperparameters ---
MAX_SEQ_LEN = 1536
NUM_EPOCHS = 1
GRAD_ACCUM = 4
LR = 5e-5
DIRECT_SAMPLES = 5000
COT_OVERSAMPLE = 2   # conservative: keep CoT as a supplement, not the dominant format
OUTPUT_DIR = '/kaggle/working/adapter'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Paths ---
TRAIN_PATH_CANDIDATES = [
    '/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv',
    '/kaggle/input/nvidia-nemotron-model-reasoning-challenge/train.csv',
    '/kaggle/input/datasets/danielgodwin/nvidia-nemotron-3-reasoning-challenge-dataset/train.csv',
]
COT_PATH_CANDIDATES = [
    '/kaggle/input/datasets/richard0307/nemotron-cot-stage2/train_cot.jsonl',
    '/kaggle/input/nemotron-cot-stage2/train_cot.jsonl',
]
BASELINE_ADAPTER_PATH = '/kaggle/input/notebooks/richard0307/nvidia-nemotron-sfttrainer-training/adapter'


def resolve_existing_path(candidates, label):
    for candidate in candidates:
        if os.path.exists(candidate):
            print(f'{label}: {candidate}')
            return candidate
    raise FileNotFoundError(f'Could not find {label}. Tried: {candidates}')


def normalize_answer(text):
    text = str(text).strip().replace(r'\ ', ' ')
    return ' '.join(text.split())


def extract_last_boxed(text):
    matches = re.findall(r'\boxed\{([^{}]+)\}', str(text))
    if not matches:
        return None
    return normalize_answer(matches[-1])


def strip_trailing_boxed(reasoning):
    reasoning = str(reasoning).strip().replace(r'\ ', ' ')
    last_idx = reasoning.rfind(r'\boxed{')
    if last_idx != -1:
        reasoning = reasoning[:last_idx].rstrip(' .??')
    return reasoning.strip()

TRAIN_PATH = resolve_existing_path(TRAIN_PATH_CANDIDATES, 'train.csv path')
COT_PATH = resolve_existing_path(COT_PATH_CANDIDATES, 'CoT path')
print(f'Baseline adapter path: {BASELINE_ADAPTER_PATH}')


## Data Loading and CoT Cleaning

The mixed dataset is built in a conservative way:
- remove any direct-answer rows whose ids already appear in CoT
- sample 5000 direct-answer rows after deduplication
- keep only CoT rows whose boxed answer still matches the ground-truth answer after normalization
- strip the trailing boxed answer from CoT reasoning before training


In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
print(f'Total train.csv samples: {len(train_df)}')

cot_rows_raw = []
with open(COT_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            cot_rows_raw.append(json.loads(line))
print(f'Loaded raw CoT samples: {len(cot_rows_raw)}')

cot_rows = []
rejected_cot = 0
for row in cot_rows_raw:
    prompt = str(row.get('prompt', '')).strip()
    answer = normalize_answer(row.get('answer', ''))
    reasoning = str(row.get('reasoning', '')).strip()
    boxed = extract_last_boxed(reasoning)
    if not prompt or not answer or not reasoning:
        rejected_cot += 1
        continue
    if boxed != answer:
        rejected_cot += 1
        continue
    clean_reasoning = strip_trailing_boxed(reasoning)
    if not clean_reasoning:
        rejected_cot += 1
        continue
    cot_rows.append({
        'id': row.get('id'),
        'prompt': prompt,
        'answer': answer,
        'reasoning': clean_reasoning,
        'boxed_answer': boxed,
    })

print(f'Accepted cleaned CoT samples: {len(cot_rows)}')
print(f'Rejected CoT samples: {rejected_cot}')

cot_ids = set(r['id'] for r in cot_rows if r.get('id') is not None)
train_df = train_df[~train_df['id'].isin(cot_ids)]
print(f'Train rows after excluding CoT ids: {len(train_df)}')

direct_count = min(DIRECT_SAMPLES, len(train_df))
direct_df = train_df.sample(n=direct_count, random_state=SEED)
print(f'Sampled direct-answer rows: {len(direct_df)}')


## Model and Tokenizer Download


In [ ]:
MODEL_PATH = kagglehub.model_download('metric/nemotron-3-nano-30b-a3b-bf16/transformers/default')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print('Tokenizer loaded.')


## Format and Mix Training Data

Important formatting decisions:
- direct-answer rows keep the original Stage 1 target format: `\boxed{answer}`
- CoT rows use `reasoning + boxed answer`, with no `<think>` tags
- both branches use the same chat template as Stage 1 whenever available


In [ ]:
def make_text(user_msg, assistant_msg):
    try:
        messages = [
            {'role': 'user', 'content': user_msg},
            {'role': 'assistant', 'content': assistant_msg},
        ]
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
    except Exception:
        return (
            f'<|im_start|>user\n{user_msg}<|im_end|>\n'
            f'<|im_start|>assistant\n{assistant_msg}<|im_end|>'
        )


def build_direct_text(row):
    prompt = str(row['prompt']).strip()
    answer = normalize_answer(row['answer'])
    user_msg = prompt + '\nPut your final answer inside \\boxed{}.'
    assistant_msg = f'\\boxed{{{answer}}}'
    return make_text(user_msg, assistant_msg)


def build_cot_text(row):
    prompt = row['prompt']
    reasoning = row['reasoning']
    answer = row['answer']
    user_msg = prompt + '\nPut your final answer inside \\boxed{}.'
    assistant_msg = f'{reasoning}\n\n\\boxed{{{answer}}}'
    return make_text(user_msg, assistant_msg)


direct_texts = [build_direct_text(row) for _, row in direct_df.iterrows()]
cot_texts = [build_cot_text(row) for row in cot_rows for _ in range(COT_OVERSAMPLE)]

all_texts = [{'text': text} for text in direct_texts + cot_texts]
random.seed(SEED)
random.shuffle(all_texts)
hf_dataset = Dataset.from_list(all_texts)

print(f'Direct samples used: {len(direct_texts)}')
print(f'CoT samples used after oversample x{COT_OVERSAMPLE}: {len(cot_texts)}')
print(f'Total mixed samples: {len(hf_dataset)}')
print(f'Estimated optimizer steps: {len(hf_dataset) // GRAD_ACCUM}')

print('\n--- Direct sample preview ---')
print(direct_texts[0][:500])
print('\n--- CoT sample preview ---')
print(cot_texts[0][:700])


## Load the Baseline Adapter for True Continue-Training

This notebook does **not** merge the baseline adapter and does **not** create a fresh LoRA.
It loads the existing Stage 1 adapter directly and keeps it trainable.


In [ ]:
for _mod_name in [
    'cutlass', 'cutlass.cute',
    'mamba_ssm',
    'mamba_ssm.modules',
    'mamba_ssm.modules.mamba3',
    'mamba_ssm.ops',
    'mamba_ssm.ops.cute',
    'mamba_ssm.ops.cute.mamba3',
    'mamba_ssm.ops.cute.mamba3.mamba3_step_fn',
    'mamba_ssm.ops.triton',
    'mamba_ssm.ops.triton.layernorm_gated',
    'causal_conv1d',
]:
    sys.modules[_mod_name] = types.ModuleType(_mod_name)

sys.modules['mamba_ssm.modules.mamba3'].Mamba3 = None
sys.modules['mamba_ssm.ops.triton.layernorm_gated'].rmsnorm_fn = purermsnorm_fn

import transformers.utils.import_utils as _iu
_iu.is_mamba_2_ssm_available = lambda: False
_iu.is_causal_conv1d_available = lambda: False

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
print(f'Base model loaded. Vocab size: {len(tokenizer)}')

for name, mod in sys.modules.items():
    if 'modeling_nemotron_h' in name:
        mod.is_fast_path_available = False
        print(f'Patched {name}: is_fast_path_available = False')

model = PeftModel.from_pretrained(
    model,
    BASELINE_ADAPTER_PATH,
    is_trainable=True,
)
model.print_trainable_parameters()
print('\nReady for continue-training on the original baseline adapter.')


## Training


In [ ]:
import triton.backends.nvidia.compiler as nv_compiler
os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = '/tmp/ptxas-blackwell'
nv_compiler.get_ptxas_version = lambda arch: '12.0'

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    logging_steps=10,
    bf16=True,
    max_grad_norm=1.0,
    optim='adamw_torch',
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    save_strategy='no',
    report_to='none',
    dataset_text_field='text',
    max_length=MAX_SEQ_LEN,
    packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': True},
)

trainer = SFTTrainer(
    model=model,
    train_dataset=hf_dataset,
    processing_class=tokenizer,
    args=training_args,
)

print(f'Starting Stage 2c revised training: {len(hf_dataset)} samples, LR={LR}, max_length={MAX_SEQ_LEN}, packing=False')
trainer.train()


## Save and Package Submission


In [ ]:
trainer.model.save_pretrained(OUTPUT_DIR)
print(f'Adapter saved to {OUTPUT_DIR}:')
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f'  {f} ({size/1024:.1f} KB)')

zip_path = '/kaggle/working/submission.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(OUTPUT_DIR):
        fpath = os.path.join(OUTPUT_DIR, fname)
        zf.write(fpath, fname)

print(f'\nCreated {zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB)')
with zipfile.ZipFile(zip_path, 'r') as zf:
    names = zf.namelist()
    print(f'Contents: {names}')
    assert 'adapter_config.json' in names
    print('submission.zip ready to submit!')
